# Risk/UQ Paper Track: Paper Tables and Figures

## Objective
Export manuscript-ready tables/figures from persisted benchmark artifacts with deterministic file paths.


## Methodology
- Reuse benchmark artifacts from `run_prefix`.
- Export canonical tables plus figures (if matplotlib available).
- Keep this notebook read-mostly: no expensive recomputation here.


## Step 1 - Bootstrap Environment

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 2 - Configure Run Identity

In [ ]:
from src.workflows import initialize_run_context, report_run_context

# Leave RUN_TAG empty to auto-adopt latest matching run under PERSIST_ROOT.
RUN_TAG = ''
RUN_TAG_PREFIX = 'risk_uq'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1'
RUN_MODE = 'auto'  # auto | fresh | resume

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=1,
    shard_id=0,
    auto_run_main_loop_when_ready=False,
    run_main_loop_override=False,
    run_tag_prefix=RUN_TAG_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=True,
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
run_prefix = run_context.run_prefix
print('run_prefix =', run_prefix)
report_run_context(run_context, display_fn=display)


## Step 3 - Export Tables/Figures

In [ ]:
from src.workflows import export_paper_tables_and_figures

EXPORT_DIR = f"{cfg.run_prefix}_paper_exports"
paper_bundle = export_paper_tables_and_figures(
    run_prefix=cfg.run_prefix,
    output_dir=EXPORT_DIR,
)

paper_bundle.exported_paths


In [ ]:
import pandas as pd
from pathlib import Path

export_dir = Path(paper_bundle.output_dir)
calib = pd.read_csv(export_dir / 'calibration_table.csv')
shift = pd.read_csv(export_dir / 'shift_robustness_table.csv')
calib.head(), shift.head()


## Report-Style Notes (Fill Before Submission)
### Claims and Evidence Mapping
- Claim 1 (calibration): table/figure references:
- Claim 2 (shift robustness): table/figure references:
- Claim 3 (decision tradeoff): table/figure references:

### Final QA
- values match exported CSVs exactly:
- all paths archived with run tag:
